In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the predictions data
df = pd.read_csv('predictions.csv')

df['max_confidence'] = df[['confidence_Problem', 'confidence_Solution', 'confidence_Other']].max(axis=1)
# Preview the data
df.head()

,tweet_id,label,confidence_Problem,confidence_Solution,confidence_Other,max_confidence
0,1433508809037783049,Problem,0.960938,0.027466,0.009888,0.960938
1,1432892478181580801,Problem,0.968750,0.019165,0.012573,0.968750
2,1433604280137699330,Problem,0.949219,0.040771,0.008545,0.949219
3,1433574214133551120,Problem,0.964844,0.024170,0.010254,0.964844
4,1538990228333002753,Problem,0.886719,0.018066,0.096680,0.886719


In [4]:
import pyreadr

# Load the Rdata file
result = pyreadr.read_r('raw_tweets (1).Rdata')

raw_tweets = result['raw_tweets']
raw_tweets.head()

,user_username,tweet_id,text,author_id,lang,created_at,in_reply_to_user_id,retweet_count,like_count,quote_count,user_tweet_count,user_list_count,sourcetweet_type,sourcetweet_text,sourcetweet_author_id,tw_date,icpsr_sen_id,state_abbrev
0,AlexPadilla4CA,1433508809037783049,Believe it: California could potentially soon ...,82489457,en,2021-09-02,NaN,1203,2917,111,14996,971,NaN,NaN,NaN,2021-09-02,42104,CA
1,AlexPadilla4CA,1432892478181580801,"All around the country, the politicization of ...",82489457,en,2021-09-01,NaN,38,128,9,14996,971,quoted,"Alabama, Georgia, Texas, Florida and Arkansas ...",759251,2021-09-01,42104,CA
2,AlexPadilla4CA,1433604280137699330,RT @BernieSanders: How does this sit with you?...,82489457,en,2021-09-03,NaN,1998,0,0,14996,971,retweeted,"How does this sit with you? In America today, ...",216776631,2021-09-03,42104,CA
3,AlexPadilla4CA,1433574214133551120,Your regular reminder that the climate crisis ...,82489457,en,2021-09-02,NaN,182,715,6,14996,971,NaN,NaN,NaN,2021-09-02,42104,CA
4,AlexPadilla4CA,1538990228333002753,The 2028 Olympics present a once-in-a-generati...,82489457,en,2022-06-20,NaN,10,78,1,14996,971,NaN,NaN,NaN,2022-06-20,42104,CA


In [5]:
# Ensure tweet_id is same type in both dataframes (int vs object causes merge error)
df['tweet_id'] = df['tweet_id'].astype(str)
raw_tweets['tweet_id'] = raw_tweets['tweet_id'].astype(str)

# Merge on tweet_id with indicator to track unmatched rows
merged = df.merge(raw_tweets, on='tweet_id', how='outer', indicator=True)

# Count non-joins from each set
not_in_raw_tweets = (merged['_merge'] == 'left_only').sum()
not_in_predictions = (merged['_merge'] == 'right_only').sum()

print(f"Predictions not in raw_tweets: {not_in_raw_tweets}")
print(f"Raw_tweets not in predictions: {not_in_predictions}")
print(f"Predictions and Raw Size: {len(df)}, {len(raw_tweets)}")

Predictions not in raw_tweets: 0
Raw_tweets not in predictions: 433
Predictions and Raw Size: 1681592, 1682025


In [6]:
# Load demographics: party_code 200=Republican, 100=Democrat, otherwise Independent
# male_1: 1=Male, 0=Female | white_dummy: 1=White, 0=Non-White
demographics = pd.read_csv('Senator_Demographics.csv')
demographics['sen_id'] = demographics['icpsr_sen_id']

# Add readable labels
demographics['party'] = demographics['party_code'].map({100: 'Democrat', 200: 'Republican'}).fillna('Independent')
demographics['gender'] = demographics['male_1'].map({1: 'Male', 0: 'Female'})
demographics['race'] = demographics['white_dummy'].map({1: 'White', 0: 'Non-White'})
demographics = demographics[['sen_id', 'state_abbrev', 'party', 'gender', 'race']]
demographics.head()

,sen_id,state_abbrev,party,gender,race
0,14400,HI,Democrat,Male,Non-White
1,40304,TN,Republican,Male,White
2,41106,NH,Republican,Female,White
3,29940,WI,Democrat,Female,White
4,40707,WY,Republican,Male,White


In [7]:
# Join demographics to merged on icpsr_sen_id
merged_df = merged.merge(demographics, left_on='icpsr_sen_id', right_on='sen_id', how='inner', indicator='demo_merge')
# Count tweets that didn't join to demographics
not_in_demographics = (merged_df['demo_merge'] == 'left_only').sum()
print(f"Tweets not in demographics: {not_in_demographics}")
merged_df.head()

print(merged_df.columns)

Tweets not in demographics: 0
Index(['tweet_id', 'label', 'confidence_Problem', 'confidence_Solution',
       'confidence_Other', 'max_confidence', 'user_username', 'text',
       'author_id', 'lang', 'created_at', 'in_reply_to_user_id',
       'retweet_count', 'like_count', 'quote_count', 'user_tweet_count',
       'user_list_count', 'sourcetweet_type', 'sourcetweet_text',
       'sourcetweet_author_id', 'tw_date', 'icpsr_sen_id', 'state_abbrev_x',
       '_merge', 'sen_id', 'state_abbrev_y', 'party', 'gender', 'race',
       'demo_merge'],
      dtype='object')


In [ ]:
# Subset to columns needed for visualizations
viz = merged_df[[
    'tweet_id', 'text',
    'confidence_Problem', 'confidence_Solution', 'confidence_Other',
    'label', 'max_confidence',
    'party', 'gender', 'race',
    'state_abbrev_y',
    'user_username', 'tw_date', 'retweet_count', 'like_count',
    'sen_id'
]].copy()

viz = viz.rename(columns={'state_abbrev_y': 'state', "user_username": "user_handle"})
viz.to_csv('final_analytics.csv')